# Lab 6 — Polygenic Scores
**Sociogenomics 2025/2026 · University of Bologna · Prof. Nicola Barban**

This notebook covers the **R analysis component** of Lab 6. All the upstream PLINK and PRSice commands (QC, clumping, scoring) are run separately in **[Google Cloud Shell](https://shell.cloud.google.com/)** following the [lab markdown](https://nicolabarban.com/sociogenomics_2025_2026/labs/week6/lab6).

This notebook loads the **pre-computed PGS outputs** from the course repository so you can focus on the statistical analysis: incremental $R^2$, bootstrap confidence intervals, and cross-ancestry portability.

---

The data files are loaded automatically from the course GitHub repository. Just run the cells in order.

In [ ]:
# Download pre-computed PGS results from the course repository
url <- "https://github.com/nicolabarban/sociogenomics_2025_2026/raw/gh_pages/labs/week6/lab6_results.zip"
download.file(url, destfile = "lab6_results.zip", mode = "wb")
unzip("lab6_results.zip", overwrite = TRUE)
list.files(pattern = "^(Trait2|FTO)")

In [ ]:
# Download phenotype, PCs and population info from the course repository
base <- "https://raw.githubusercontent.com/nicolabarban/sociogenomics_2025_2026/gh_pages/data/"
for (f in c("1kg.Trait2.phen", "1kg_pca.eigenvec", "1kg-sample-2504-phased.txt", "BMI_pheno.txt")) {
  download.file(paste0(base, f), destfile = f, quiet = TRUE)
}
list.files(pattern = "\\.(phen|eigenvec|txt)$")

## Part III — Monogenic FTO score

The score file `score_rs9930506.txt` contains a single SNP (rs9930506 in *FTO*) with effect size 0.4 kg/m² per A allele.

The file `FTOscore.profile` was produced by PLINK `--score`. Let's inspect it.

In [ ]:
fto <- read.table("FTOscore.profile", header = TRUE)
head(fto)
table(fto$SCORE)

## Part VI — Analyse the PGS in R

We load the PGS at different $p$-value thresholds (from PRSice), the phenotype, and the principal components, then fit a linear model adjusting for population structure.

In [ ]:
# Load the PGS at each threshold produced by PRSice
all_scores <- read.table("Trait2_PRSice.all_score", header = TRUE)
head(all_scores)
cat("\nColumns in the score file:\n")
print(colnames(all_scores))

> **Note:** R replaces `-` with `.` in column names, so `Pt_5e-08` becomes `Pt_5e.08`, and `Pt_5e-04` becomes `Pt_0.0005`.

In [ ]:
# Load phenotype, PCs and merge
pheno <- read.table("1kg.Trait2.phen", header = FALSE,
                    col.names = c("FID", "IID", "Trait2"))

pca_cols <- c("FID", "IID", paste0("PC", 1:10))
pca <- read.table("1kg_pca.eigenvec", header = FALSE, col.names = pca_cols)

d <- merge(all_scores, pheno, by = c("FID", "IID"))
d <- merge(d, pca, by = c("FID", "IID"))

cat("Rows after merge:", nrow(d), "\n")
head(d[, c("IID", "Trait2", "Pt_0.5", "PC1", "PC2")])

In [ ]:
# Standardise the PGS at the best threshold (Pt_0.5 from PRSice)
d$PGS_best <- scale(d$Pt_0.5)
d$PGS_gw   <- scale(d$Pt_5e.08)
d$PGS_all  <- scale(d$Pt_1)

summary(d$PGS_best)

### PGS distribution

By the Central Limit Theorem, the PGS is approximately normally distributed.

In [ ]:
hist(d$PGS_best, breaks = 30, col = "steelblue", border = "white",
     main = "Distribution of the standardised PGS",
     xlab = "PGS (z-score)")

### Incremental $R^2$

We compare two models:

- **Baseline:** only 10 principal components (capturing population structure)
- **Full:** PCs + PGS

The difference in $R^2$ is the **incremental** variance explained by the PGS, net of ancestry.

In [ ]:
mod0 <- lm(Trait2 ~ PC1 + PC2 + PC3 + PC4 + PC5 +
                     PC6 + PC7 + PC8 + PC9 + PC10, data = d)
mod1 <- lm(Trait2 ~ PGS_best + PC1 + PC2 + PC3 + PC4 + PC5 +
                    PC6 + PC7 + PC8 + PC9 + PC10, data = d)

delta_r2 <- summary(mod1)$r.squared - summary(mod0)$r.squared
cat("R2 baseline (PCs only):", round(summary(mod0)$r.squared, 4), "\n")
cat("R2 full (PCs + PGS):   ", round(summary(mod1)$r.squared, 4), "\n")
cat("Incremental R2:        ", round(delta_r2, 4), "\n")

summary(mod1)$coefficients["PGS_best", ]

### Compare $R^2$ across $p$-value thresholds

In [ ]:
thresholds <- c("Pt_5e.08", "Pt_5e.06", "Pt_0.0005", "Pt_0.05", "Pt_0.5", "Pt_1")
labels     <- c("5e-8", "5e-6", "5e-4", "0.05", "0.5", "1")

results <- data.frame(threshold = labels, delta_r2 = NA)
r2_base <- summary(mod0)$r.squared

for (i in seq_along(thresholds)) {
  d$tmp <- scale(d[[thresholds[i]]])
  mod   <- lm(Trait2 ~ tmp + PC1 + PC2 + PC3 + PC4 + PC5 +
                        PC6 + PC7 + PC8 + PC9 + PC10, data = d)
  results$delta_r2[i] <- summary(mod)$r.squared - r2_base
}

print(results)

In [ ]:
bp <- barplot(results$delta_r2, names.arg = results$threshold,
              col = "steelblue", border = NA,
              ylim = c(0, max(results$delta_r2) * 1.2),
              main = "Incremental R² by p-value threshold",
              xlab = "p-value threshold", ylab = "Incremental R²")
text(bp, results$delta_r2 + 0.005, round(results$delta_r2, 3), cex = 0.8)

### Bootstrap 95% confidence interval for $\Delta R^2$

In [ ]:
library(boot)
set.seed(12345)

rsq_fn <- function(data, indices) {
  ds <- data[indices, ]
  m0 <- lm(Trait2 ~ PC1 + PC2 + PC3 + PC4 + PC5 +
                    PC6 + PC7 + PC8 + PC9 + PC10, data = ds)
  m1 <- lm(Trait2 ~ PGS_best + PC1 + PC2 + PC3 + PC4 + PC5 +
                    PC6 + PC7 + PC8 + PC9 + PC10, data = ds)
  summary(m1)$r.squared - summary(m0)$r.squared
}

results_boot <- boot(data = d, statistic = rsq_fn, R = 1000)
boot.ci(results_boot, type = "norm")

## Part VII — Cross-ancestry portability

Now we take the same PGS, apply it to **all** 1000 Genomes super-populations, and compare the $R^2$.

In [ ]:
pgs_all <- read.table("Trait2_pgs_all_pops.profile", header = TRUE)
pop     <- read.table("1kg-sample-2504-phased.txt", header = TRUE)

da <- merge(pgs_all[, c("FID", "IID", "SCORE")], pheno,
            by = c("FID", "IID"))
da <- merge(da, pop[, c("sample", "super_pop")],
            by.x = "IID", by.y = "sample")

cat("Individuals per super-population:\n")
print(table(da$super_pop))

In [ ]:
for (p in c("EUR", "EAS", "SAS", "AFR", "AMR")) {
  sub <- da[da$super_pop == p, ]
  if (nrow(sub) == 0) {
    cat(sprintf("%s: no samples\n", p))
    next
  }
  r2 <- cor(sub$SCORE, sub$Trait2)^2
  cat(sprintf("%s: R² = %.4f  (N = %d)\n", p, r2, nrow(sub)))
}

In [ ]:
# Distribution of the PGS itself across ancestries
boxplot(SCORE ~ super_pop, data = da,
        col = "lightblue",
        main = "PGS distribution by super-population",
        xlab = "Super-population", ylab = "Raw PGS (PLINK --score)")

---

## Summary

| Result | Value |
|---|---|
| Best $p$-value threshold (PRSice) | 0.5 |
| Incremental $\Delta R^2$ (with PCs) | ~14.7% |
| 95% bootstrap CI | ~(9%, 21%) |
| Cross-ancestry relative accuracy (AFR vs.\ EUR) | ~30% |

Key take-aways:

- PGS predicts the trait much better than PCs alone, but even the best PGS explains only a fraction of the phenotypic variance.
- Incremental $R^2$ plateaus around $p < 0.5$ --- most of the signal comes from SNPs that are not genome-wide significant.
- The PGS is systematically **less accurate** in non-European populations. Population stratification, LD differences, and effect-size heterogeneity all contribute.